# Data preprocessing

In order to develop a proper model for the estimation of the surface water fracture, we first must adapt the data of previous satelites so that they can be used in our model.

### Library loading

Due to discrepancies between the rasterio and gdal libraries, we'll be using two different environments in this notebook.

#### General libraries

These are the necessary libraries to run most of the code.

In [1]:
import xarray as xr
import numpy as np
import os
import zipfile
import io
import re
import requests

from collections import defaultdict
from pathlib import Path
from tqdm import tqdm
from typing import List
from datetime import datetime, date, timedelta

time_coder = xr.coders.CFDatetimeCoder(use_cftime=True)

#### gdal library

The gdal library is necessary for the reprojection part of the notebook.

In [5]:
import os
import re
from osgeo import gdal, osr

ModuleNotFoundError: No module named 'osgeo'

## Data download

We'll start by manually downloading all of the [WINDSAT](https://data.remss.com/TB/intercalibration/windsat_TB_maps_daily_025deg_unfiltered/) satellite data. The [LPDR](http://files.ntsg.umt.edu/data/LPDR_v3/GeoTif/2017/) data can be downloaded directly without issues.

Note that the download speed is excrutiengly long (around 15 minutes per file). It is recomended to download it in chuncks, progressively moving the `end_date` further away, or obtain the data in any other way.

In [8]:
BASE_URL = (
    "https://data.remss.com/TB/intercalibration/"
    "windsat_TB_maps_daily_025deg_unfiltered/"
)

OUTPUT_DIR = os.path.join("data", "WINDSAT")
os.makedirs(OUTPUT_DIR, exist_ok=True)

start_date = date(2017, 1, 1)
end_date = date(2017, 12, 31)

current_date = start_date

while current_date <= end_date:
    datestr = current_date.strftime("%Y_%m_%d")
    filename = f"RSS_WINDSAT_DAILY_TBTOA_MAPS_{datestr}.nc"
    url = BASE_URL + filename
    local_path = os.path.join(OUTPUT_DIR, filename)

    if os.path.exists(local_path):
        print(f"Skipping (already exists): {filename}")
        current_date += timedelta(days=1)
        continue

    print(f"Downloading: {filename}")

    try:
        response = requests.get(url, stream=True, timeout=60)
        response.raise_for_status()

        with open(local_path, "wb") as f:
            for chunk in response.iter_content(chunk_size=8192):
                if chunk:
                    f.write(chunk)

    except requests.RequestException as e:
        print(f"Failed to download {filename}: {e}")

    current_date += timedelta(days=1)

print("Download process completed.")

Skipping (already exists): RSS_WINDSAT_DAILY_TBTOA_MAPS_2017_01_01.nc
Skipping (already exists): RSS_WINDSAT_DAILY_TBTOA_MAPS_2017_01_02.nc
Skipping (already exists): RSS_WINDSAT_DAILY_TBTOA_MAPS_2017_01_03.nc
Skipping (already exists): RSS_WINDSAT_DAILY_TBTOA_MAPS_2017_01_04.nc
Skipping (already exists): RSS_WINDSAT_DAILY_TBTOA_MAPS_2017_01_05.nc
Skipping (already exists): RSS_WINDSAT_DAILY_TBTOA_MAPS_2017_01_06.nc
Skipping (already exists): RSS_WINDSAT_DAILY_TBTOA_MAPS_2017_01_07.nc
Skipping (already exists): RSS_WINDSAT_DAILY_TBTOA_MAPS_2017_01_08.nc
Skipping (already exists): RSS_WINDSAT_DAILY_TBTOA_MAPS_2017_01_09.nc
Skipping (already exists): RSS_WINDSAT_DAILY_TBTOA_MAPS_2017_01_10.nc
Skipping (already exists): RSS_WINDSAT_DAILY_TBTOA_MAPS_2017_01_11.nc
Skipping (already exists): RSS_WINDSAT_DAILY_TBTOA_MAPS_2017_01_12.nc
Skipping (already exists): RSS_WINDSAT_DAILY_TBTOA_MAPS_2017_01_13.nc
Skipping (already exists): RSS_WINDSAT_DAILY_TBTOA_MAPS_2017_01_14.nc
Skipping (already ex

## Data analysis

Now that we have all of our data, let's check it's structure and see if it's properly formated.

### LPDR

In [23]:
lpdr_ds = xr.open_dataset("data/LPDR/2017tif/AMSRU_Mland_2017001A.tif")

print("Dimensions:", dict(lpdr_ds.sizes))
print("Coordinates:")
for coord in ["x", "y"]:
    if coord in lpdr_ds.coords:
        print(f"  {coord}: {lpdr_ds.coords[coord].values[0]}  →  {lpdr_ds.coords[coord].values[-1]}")

Dimensions: {'band': 7, 'x': 1383, 'y': 586}
Coordinates:
  x: -17321659.775000002  →  17321659.77499638
  y: 7332251.062494965  →  -7332251.0625035055


The LPDR dataset has an EASE-v2 projection.

### Windsat bands

In [31]:
windsat_ds = xr.open_dataset("data/WINDSAT/RSS_WINDSAT_DAILY_TBTOA_MAPS_2017_01_01.nc", decode_times=time_coder)
windsat_ds = windsat_ds.set_coords(["longitude", "latitude"])

print("Dimensions:", dict(windsat_ds.sizes))
print("Coordinates:")
for coord in ["longitude", "latitude"]:
    if coord in windsat_ds.coords:
        print(f"  {coord}: {windsat_ds.coords[coord].values[0]}  →  {windsat_ds.coords[coord].values[-1]}")

Dimensions: {'longitude_grid': 1440, 'latitude_grid': 720, 'swath_sector': 2, 'look_direction': 2, 'frequency_band': 5, 'polarization': 4, 'polarization_dual': 2}
Coordinates:
  longitude: 0.125  →  359.875
  latitude: -89.875  →  89.875


The WINDSAT dataset has a simple EQR projection.

## Data projection

Since both datasets do not share the same grid, we need to reproject them to a common grid. We'll do this by reprojecting the LPDR dataset to the same projection as the Windsat dataset.

In [5]:
def reproject_file(file_path: str, output_folder: str = None) -> bool:
    """
        Read the input geotiff in EASE v1
        Reproject + resample into ED 0.25º
        create latitude an longitude bands for convenience.
        (bands added as second to last, and last band respectively)

        param output_folder: name of the new file to save reprojected data.
            If None, data will be ovewritten in file_path.

        Returns bool: whether or not the file was succesfully reprojected
    """
    dataset = gdal.Open(file_path)

    # Define the geotransform for the output dataset
    target_geotransform = (0, 0.25, 0.0, 90.0, 0.0, -0.25)

    output_width = 1440
    output_height = 720

    # Define src and geotransform from the input:
    source_srs = osr.SpatialReference()
    source_srs.ImportFromEPSG(3410)

    # Desired projection
    target_srs = osr.SpatialReference()
    target_srs.ImportFromEPSG(4326)

    # Declare the output file and driver:
    driver = gdal.GetDriverByName("GTiff")

    if output_folder is None:
        output_file = os.path.join(os.path.dirname(file_path), "temp.tif")

    else:
        output_file = os.path.join(output_folder, os.path.basename(file_path))

    # Create the new dataset
    output_dataset = driver.Create(
        output_file, output_width, output_height,
        dataset.RasterCount,
        gdal.GDT_Float32
    )
    output_dataset.SetProjection(target_srs.ExportToWkt())
    output_dataset.SetGeoTransform(target_geotransform)

    # Set the output dataset value to -999.0, instead of 0.
    for i in range(1, output_dataset.RasterCount, 1):
        band = output_dataset.GetRasterBand(i)
        arr = band.ReadAsArray()
        arr = arr - 999.0
        band.WriteArray(arr)

    # Reproject and resample using gdal.Warp()
    gdal.Warp(
        output_dataset,
        dataset,
        dstSRS=target_srs.ExportToWkt(),
        width=output_width,
        height=output_height,
        resampleAlg="near",
        # GRA_Bilinear TODO: bilinear warp does not work with Nodata params,
        # there will be values between -999 and the valid range
        srcNodata=float(-999.0),
        dstNodata=-999.0,
    )

    # Close the files
    dataset = None
    output_dataset = None

    if output_folder is None:
        # Delete original file, rename temp file.
        os.remove(file_path)
        os.rename(output_file, file_path)

    return True

source_folder = 'data/LPDR/2017tif'
output_folder = 'data/LPDR/2017tifrep'

print("START")

# NOTE: Do not reproject Quality Flag files for now,
# those files end in '\d{3}QA.tif'
# Select only ascending and descending passes:
regex = r"\d{7}[AD].tif"

for file_name in os.listdir(source_folder):
    file_path = os.path.join(source_folder, file_name)
    print(f"Reprojecting {file_path}")

    outcome = reproject_file(file_path, output_folder=output_folder)

    if outcome:
        print("Reprojected")
        if output_folder is not None:
            print(
                f"""New file saved {
                    os.path.join(output_folder, file_name)
                }"""
            )
    else:
        print(f"File {file_name} is already reprojected")

print("DONE")

START
Reprojecting data/LPDR/2017tif\AMSRU_Mland_2017001A.tif
Reprojected
New file saved data/LPDR/2017tifrep\AMSRU_Mland_2017001A.tif
Reprojecting data/LPDR/2017tif\AMSRU_Mland_2017001A_QA.tif
Reprojected
New file saved data/LPDR/2017tifrep\AMSRU_Mland_2017001A_QA.tif
Reprojecting data/LPDR/2017tif\AMSRU_Mland_2017001D.tif
Reprojected
New file saved data/LPDR/2017tifrep\AMSRU_Mland_2017001D.tif
Reprojecting data/LPDR/2017tif\AMSRU_Mland_2017001D_QA.tif
Reprojected
New file saved data/LPDR/2017tifrep\AMSRU_Mland_2017001D_QA.tif
Reprojecting data/LPDR/2017tif\AMSRU_Mland_2017002A.tif
Reprojected
New file saved data/LPDR/2017tifrep\AMSRU_Mland_2017002A.tif
Reprojecting data/LPDR/2017tif\AMSRU_Mland_2017002A_QA.tif
Reprojected
New file saved data/LPDR/2017tifrep\AMSRU_Mland_2017002A_QA.tif
Reprojecting data/LPDR/2017tif\AMSRU_Mland_2017002D.tif
Reprojected
New file saved data/LPDR/2017tifrep\AMSRU_Mland_2017002D.tif
Reprojecting data/LPDR/2017tif\AMSRU_Mland_2017002D_QA.tif
Reprojected
Ne

Let's check the new files to see if the reprojection worked accordingly.

### LPDR reprojected

In [2]:
lpdr_rep_ds = xr.open_dataset("data/LPDR/2017tifrep/AMSRU_Mland_2017001A.tif")

lpdr_rep_ds = lpdr_rep_ds.rename({
    "x": "longitude",
    "y": "latitude"
})
lpdr_rep_ds = lpdr_rep_ds.sortby("latitude")

print("Dimensions:", dict(lpdr_rep_ds.sizes))
print("Coordinates:")
for coord in ["longitude", "latitude"]:
    if coord in lpdr_rep_ds.coords:
        print(f"  {coord}: {lpdr_rep_ds.coords[coord].values[0]}  →  {lpdr_rep_ds.coords[coord].values[-1]}")

Dimensions: {'band': 7, 'longitude': 1440, 'latitude': 720}
Coordinates:
  longitude: 0.125  →  359.875
  latitude: -89.875  →  89.875


As we can see, the reprojection worked perfectly, and we can move on to the next step.

## Data mixing

Since all datasets now follow the same grid, we can go ahead and combine all three of them into a single training dataset.

### File pairing

Before merging the proper datasets together, we have to group each file by the day of the year.

In [2]:
def get_day_of_year(file_path):
    file_path = str(file_path)
    if 'AMSRU' in file_path:
        year_doyy_str = os.path.basename(file_path).split('_')[2][:7]
        year_doyy = year_doyy_str[-3:]
    elif 'WINDSAT' in file_path:
        year, month, day = os.path.basename(file_path).split("_")[-3:]
        day = day.replace(".nc", "")
        date = datetime(int(year), int(month), int(day))
        year_doyy = f"{date.timetuple().tm_yday:03d}"
    return year_doyy

def group_files_by_day(lpdr_dir: str, windsat_dir: str,) -> List[List[str]]:
    # day -> {"lpdr": [...], "windsat": [...]}
    grouped = defaultdict(lambda: {"lpdr": [], "windsat": []})

    # Collect LPDR files
    for path in Path(lpdr_dir).iterdir():
        if path.is_file():
            doy = get_day_of_year(path)
            grouped[doy]["lpdr"].append(str(path))

    # Collect WINDSAT files
    for path in Path(windsat_dir).iterdir():
        if path.is_file():
            doy = get_day_of_year(path)
            grouped[doy]["windsat"].append(str(path))

    # Build final list, sorted by DOY
    output = []
    for doy in sorted(grouped.keys()):
        lpdr_files = sorted(grouped[doy]["lpdr"])
        windsat_files = sorted(grouped[doy]["windsat"])

        # Optional sanity checks (can be removed if not desired)
        if len(lpdr_files) > 4:
            raise ValueError(f"More than 4 LPDR files for DOY {doy}")
        if len(windsat_files) > 1:
            raise ValueError(f"More than 1 WINDSAT file for DOY {doy}")

        slot = lpdr_files + windsat_files
        output.append(slot)

    return output

In [3]:
lpdr_dir = "data/LPDR/2017tifrep/"
windsat_dir = "data/WINDSAT/"

corresponding_files = group_files_by_day(lpdr_dir, windsat_dir)

### Dataset unification

For every day there are four files of LPDR data, one full dataset for every swath_sector (Ascending and Descending) and two quality flags for every respecting swath_sector.

We'll achieve a unified dataset by adding every variable of these four files into the WINDSAT one.

In [4]:
AMSRU_VARIABLES = ["fwns", "fw", "air_temperature", "pwv", "vod", "vsm", "vpd"]

def load_amsru(data_path, qa_path):
    ds = xr.open_dataset(data_path)
    qa = xr.open_dataset(qa_path)

    ds = ds.rename({"x": "longitude", "y": "latitude"})
    qa = qa.rename({"x": "longitude", "y": "latitude"})

    ds = ds.sortby("latitude")
    qa = qa.sortby("latitude")

    return ds, qa

def extract_amsru_variables(ds, qa):
    data_dict = {}

    # --- Extract AMSRU variables ---
    for i, var_name in enumerate(AMSRU_VARIABLES):
        data_dict[var_name] = ds["band_data"].isel(band=i)

    # --- Extract QA (single shared field) ---
    qa_da = qa["band_data"]
    if "band" in qa_da.dims:
        qa_da = qa_da.isel(band=0)

    return data_dict, qa_da

def merge_amsru_into_windsat(windsat_ds, amsru_A, qa_A, amsru_D, qa_D):
    lat = windsat_ds.latitude
    lon = windsat_ds.longitude
    swath = windsat_ds.swath_sector

    new_vars = {}

    # --- Shared QA variable (added once) ---
    amsru_qc = xr.DataArray(
        np.full((lat.size, lon.size, swath.size), -1, dtype=np.int16),
        dims=("latitude_grid", "longitude_grid", "swath_sector"),
        coords={
            "latitude_grid": lat,
            "longitude_grid": lon,
            "swath_sector": swath
        },
        name="amsru_quality_flag",
        attrs={
            "long_name": "AMSR-U quality flag",
            "instrument": "AMSR-U",
            "platform": "Aqua",
            "source": "LPDR",
            "comment": "Shared QA flag for all AMSR-U retrieval variables"
        }
    )

    amsru_qc.loc[:, :, 0] = qa_A.values
    amsru_qc.loc[:, :, 1] = qa_D.values

    new_vars["amsru_quality_flag"] = amsru_qc

    # --- AMSRU variables ---
    for var in AMSRU_VARIABLES:
        data = xr.DataArray(
            np.full((lat.size, lon.size, swath.size), np.nan, dtype=np.float32),
            dims=("latitude_grid", "longitude_grid", "swath_sector"),
            coords=amsru_qc.coords,
            name=var,
            attrs={
                "long_name": f"AMSR-U {var.replace('_', ' ')}",
                "instrument": "AMSR-U",
                "platform": "Aqua",
                "source": "LPDR"
            }
        )

        data.loc[:, :, 0] = amsru_A[var].values
        data.loc[:, :, 1] = amsru_D[var].values

        new_vars[data.name] = data

    return windsat_ds.assign(**new_vars)

In [9]:
output_dir = Path("data/MERGED")
output_dir.mkdir(parents=True, exist_ok=True)

for files in tqdm(corresponding_files, desc="Processing days", unit="day"):
    lpdr_A_1, lpdr_A_2, lpdr_D_1, lpdr_D_2, windsat_file = files

    # Retrieve DOY from any file (they all share it)
    doy = get_day_of_year(windsat_file)

    out_file = output_dir / f"WINDSAT_AMSRU_2017_{doy}.nc"

    # --- Skip if already processed ---
    if out_file.exists():
        tqdm.write(f"[DOY {doy}] Output already exists. Skipping.")
        continue

    tqdm.write(f"[DOY {doy}] Processing")

    try:
        # --- Load AMSRU ascending ---
        ds_A, qa_A = load_amsru(
            lpdr_A_1,
            lpdr_A_2
        )

        # --- Load AMSRU descending ---
        ds_D, qa_D = load_amsru(
            lpdr_D_1,
            lpdr_D_2
        )

        # --- Load WINDSAT ---
        windsat_ds = xr.open_dataset(
            windsat_file,
            decode_times=time_coder
        ).set_coords(["longitude", "latitude"])

        # --- Extract AMSRU variables ---
        amsru_A, qaA = extract_amsru_variables(ds_A, qa_A)
        amsru_D, qaD = extract_amsru_variables(ds_D, qa_D)

        # --- Merge ---
        merged_ds = merge_amsru_into_windsat(
            windsat_ds,
            amsru_A, qaA,
            amsru_D, qaD
        )

        merged_ds = merged_ds.reset_coords(
            ["latitude", "longitude"],
            drop=True
        )

        # --- Save ---
        merged_ds.to_netcdf(
            out_file,
            format="NETCDF4"
        )

    finally:
        # --- Ensure files are closed to avoid HDF errors ---
        for ds in (locals().get("ds_A"),
                   locals().get("ds_D"),
                   locals().get("windsat_ds"),
                   locals().get("merged_ds")):
            if ds is not None:
                ds.close()

Processing days:   0%|          | 0/365 [00:00<?, ?day/s]

[DOY 001] Output already exists. Skipping.
[DOY 002] Output already exists. Skipping.
[DOY 003] Output already exists. Skipping.
[DOY 004] Output already exists. Skipping.
[DOY 005] Output already exists. Skipping.
[DOY 006] Output already exists. Skipping.
[DOY 007] Output already exists. Skipping.
[DOY 008] Output already exists. Skipping.
[DOY 009] Output already exists. Skipping.
[DOY 010] Output already exists. Skipping.
[DOY 011] Output already exists. Skipping.
[DOY 012] Output already exists. Skipping.
[DOY 013] Output already exists. Skipping.
[DOY 014] Output already exists. Skipping.
[DOY 015] Output already exists. Skipping.
[DOY 016] Output already exists. Skipping.
[DOY 017] Output already exists. Skipping.
[DOY 018] Output already exists. Skipping.
[DOY 019] Output already exists. Skipping.
[DOY 020] Output already exists. Skipping.
[DOY 021] Output already exists. Skipping.
[DOY 022] Output already exists. Skipping.
[DOY 023] Output already exists. Skipping.
[DOY 024] O

Processing days:   7%|▋         | 25/365 [02:18<31:23,  5.54s/day]

[DOY 026] Processing


Processing days:   7%|▋         | 26/365 [04:39<1:12:49, 12.89s/day]

...

Processing days:  15%|█▍        | 53/365 [1:05:51<11:49:07, 136.37s/day]

[DOY 054] Processing


Processing days:  15%|█▍        | 54/365 [1:08:08<6:32:24, 75.70s/day]  

[DOY 055] Processing


OSError: [Errno -101] NetCDF: HDF error: 'c:\\Users\\marce\\Documents\\TFM_MCD_MarcVelascoMateu\\data\\WINDSAT\\RSS_WINDSAT_DAILY_TBTOA_MAPS_2017_02_24.nc'

Let's finish off by checking that everything went well.

In [59]:
windsat_ds = xr.open_dataset("data/MERGED/WINDSAT_AMSRU_2017_01_01.nc", decode_times=time_coder)
windsat_ds

<xarray.Dataset> Size: 1GB
Dimensions:             (swath_sector: 2, look_direction: 2, frequency_band: 5,
                         latitude_grid: 720, longitude_grid: 1440,
                         polarization: 4, polarization_dual: 2)
Coordinates:
  * swath_sector        (swath_sector) int64 16B 0 1
  * latitude_grid       (latitude_grid) float32 3kB -89.88 -89.62 ... 89.88
  * longitude_grid      (longitude_grid) float32 6kB 0.125 0.375 ... 359.6 359.9
Dimensions without coordinates: look_direction, frequency_band, polarization,
                                polarization_dual
Data variables: (12/40)
    node                (swath_sector) int32 8B ...
    look                (look_direction) int32 8B ...
    frequency_vpol      (frequency_band) float32 20B ...
    frequency_hpol      (frequency_band) float32 20B ...
    eia_nominal         (frequency_band) float32 20B ...
    time                (frequency_band, latitude_grid, longitude_grid, look_direction, swath_sector) object 166MB ...
    ...                  ...
    fw                  (latitude_grid, longitude_grid, swath_sector) float32 8MB ...
    air_temperature     (latitude_grid, longitude_grid, swath_sector) float32 8MB ...
    pwv                 (latitude_grid, longitude_grid, swath_sector) float32 8MB ...
    vod                 (latitude_grid, longitude_grid, swath_sector) float32 8MB ...
    vsm                 (latitude_grid, longitude_grid, swath_sector) float32 8MB ...
    vpd                 (latitude_grid, longitude_grid, swath_sector) float32 8MB ...
Attributes: (12/71)
    Conventions:                            CF-1.7
    title:                                  RSS WindSat TOA Brightness Temper...
    version:                                V01.0
    summary:                                The dataset contains RSS WindSat ...
    references:                              [1] T. Meissner et al., Remote S...
    acknowledgement:                        Funded under NASA Grant 80NSSC21K...
    ...                                     ...
    Source_of_ancillary_NOAA_OI_SST:        NOAA OI SST V2 High Resolution Da...
    Source_of_ancillary_IMERG_rain_rate:    Huffman, G. et al.,  2019. NASA G...
    Source_of_ancillary_CCMP_wind:          Mears, C. et al., 2023.Remote Sen...
    Source_of_ancillary_HYCOM_SSS:          Hybrid Coordinate Ocean Model, GL...
    Source_of_ancillary_ERA5:               ECMWF Reanalysis v5 (ERA5). https...
    Source_of_RSS_WindSat_AS_ECV:           https://www.remss.com/missions/wi...

In [2]:
from pathlib import Path
import xarray as xr

def find_corrupted_netcdf(folder):
    corrupted = []

    for file in Path(folder).iterdir():
        if not file.is_file():
            continue

        try:
            ds = xr.open_dataset(file, decode_times=time_coder)
            ds.close()
        except Exception as e:
            corrupted.append((file.name, repr(e)))

    return corrupted


folder = r"data/WINDSAT"
bad_files = find_corrupted_netcdf(folder)

print(f"Corrupted NetCDF files ({len(bad_files)}):")
for name, err in bad_files:
    print(f"{name} -> {err}")


Corrupted NetCDF files (0):
